# Fine tuning: grid search

The model chosen in `03_models.ipynb` is **HistGradientBoosting**. Here we look for its best settings
(hyperparameters) with a **grid search**: every combination of the values below is tried, trained on
months 0–3 and graded on month 4. Month 5 (test) stays closed.

Why not scikit-learn's `GridSearchCV`: it splits the data into random pieces (cross-validation), mixing
months; we want to grade every combination on the **future** month, as in real life.

## 1. Load and grade (same as 03_models)

In [1]:
import itertools
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

df = pd.read_parquet("../data/processed/features.parquet")
info = json.load(open("features.json"))
FEATURES, CATEGORIES = info["features"], info["categories"]
for c in CATEGORIES:
    df[c] = df[c].astype("category")

train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
X_train, y_train = train[FEATURES], train["isFraud"]
X_valid, y_valid = valid[FEATURES], valid["isFraud"]

FLAG_SHARE = 0.02

def evaluate(scores):
    """Grades on the valid month: overall grades + what happens if we flag the top 2%."""
    y = y_valid.to_numpy()
    amount = valid["TransactionAmt"].to_numpy()
    flagged = np.argsort(-scores)[:int(len(y) * FLAG_SHARE)]
    return {
        "PR-AUC": round(average_precision_score(y, scores), 4),
        "ROC-AUC": round(roc_auc_score(y, scores), 4),
        "frauds caught": round(y[flagged].sum() / y.sum() * 100, 1),
        "precision": round(y[flagged].mean() * 100, 1),
        "fraud $ caught": round(amount[flagged][y[flagged] == 1].sum() / amount[y == 1].sum() * 100, 1),
    }

## 2. The grid

Every setting ("knob") with the values to try. The grid is **every combination**: 2 × 3 × 2 × 2 × 2 = 48
models, about 1 minute each → **~50 minutes**.

| knob | what it does |
|---|---|
| `learning_rate` | how much each tree corrects: small = many small steps |
| `max_leaf_nodes` | in how many groups each tree can split the purchases: more = more detailed trees |
| `min_samples_leaf` | minimum purchases in each group: higher = no tiny groups learned by heart |
| `l2_regularization` | a "fine" on strong corrections: higher = more prudent |
| `max_features` | share of columns each tree can look at: lower = more different trees |

In [2]:
GRID = {
    "learning_rate": [0.05, 0.1],
    "max_leaf_nodes": [31, 63, 127],
    "min_samples_leaf": [20, 100],
    "l2_regularization": [0.0, 1.0],
    "max_features": [0.5, 1.0],
}
combinations = [dict(zip(GRID, values)) for values in itertools.product(*GRID.values())]
print(len(combinations), "combinations to try")

48 combinations to try


## 3. Try every combination

Every result is saved to `data/processed/tuning_results.csv` as soon as it is ready. If the notebook stops
(or you stop it), run the cell again: the combinations already done are skipped.

In [3]:
RESULTS_FILE = Path("../data/processed/tuning_results.csv")
done = pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame()

for i, params in enumerate(combinations, 1):
    if len(done) and ((done[list(params)] == pd.Series(params)).all(axis=1)).any():
        continue                                                   # already tried
    model = HistGradientBoostingClassifier(
        max_iter=1000, categorical_features="from_dtype",
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=50,
        random_state=0, **params,
    )
    start = time.time()
    model.fit(X_train, y_train)
    grades = evaluate(model.predict_proba(X_valid)[:, 1])
    row = {**params, "trees": model.n_iter_, **grades, "seconds": round(time.time() - start)}
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
    done.to_csv(RESULTS_FILE, index=False)
    print(f"{i}/{len(combinations)}  PR-AUC {grades['PR-AUC']}  {params}")

1/48  PR-AUC 0.5666  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_features': 0.5}
2/48  PR-AUC 0.5698  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'max_features': 1.0}
3/48  PR-AUC 0.5672  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'max_features': 0.5}
4/48  PR-AUC 0.5672  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'max_features': 1.0}
5/48  PR-AUC 0.5763  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 100, 'l2_regularization': 0.0, 'max_features': 0.5}
6/48  PR-AUC 0.5724  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 100, 'l2_regularization': 0.0, 'max_features': 1.0}
7/48  PR-AUC 0.5679  {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'min_samples_leaf': 100, 'l2_regularization': 1.0, 'max_features': 0.5}
8/48  PR-AUC 0.5711  {'learning

## 4. The best combinations

In [4]:
results = pd.read_csv(RESULTS_FILE).sort_values("PR-AUC", ascending=False)
results.head(10)

,learning_rate,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,trees,PR-AUC,ROC-AUC,frauds caught,precision,fraud $ caught,seconds
18,0.05,127,20,1.0,0.5,1000,0.5970,0.9191,44.2,76.7,40.2,98
16,0.05,127,20,0.0,0.5,1000,0.5969,0.9174,44.2,76.8,38.7,91
22,0.05,127,100,1.0,0.5,1000,0.5960,0.9192,44.1,76.6,40.3,143
17,0.05,127,20,0.0,1.0,1000,0.5943,0.9149,44.7,77.7,40.1,89
47,0.10,127,100,1.0,1.0,654,0.5938,0.9168,44.2,76.7,38.6,72
40,0.10,127,20,0.0,0.5,644,0.5937,0.9134,44.7,77.5,40.6,54
20,0.05,127,100,0.0,0.5,977,0.5922,0.9161,44.3,76.8,38.6,89
32,0.10,63,20,0.0,0.5,1000,0.5909,0.9120,44.6,77.5,40.7,69
23,0.05,127,100,1.0,1.0,1000,0.5907,0.9152,43.9,76.2,38.2,137
21,0.05,127,100,0.0,1.0,1000,0.5899,0.9153,43.8,76.0,37.3,103


## 5. Which value of each knob works best

For every knob, the average PR-AUC of all the models that used each value. Example: if the models with
`min_samples_leaf = 100` have a higher average than those with 20, bigger groups help.

In [5]:
for knob in GRID:
    print(results.groupby(knob)["PR-AUC"].mean().round(4).to_string(), "\n")

learning_rate
0.05    0.5823
0.10    0.5825 

max_leaf_nodes
31     0.5733
63     0.5837
127    0.5901 

min_samples_leaf
20     0.5827
100    0.5820 

l2_regularization
0.0    0.5833
1.0    0.5814 

max_features
0.5    0.5828
1.0    0.5819 



## 6. Save the best settings

Used by the next notebook to train the final model.

In [6]:
best = results.iloc[0]
best_params = {k: (int(best[k]) if k in ("max_leaf_nodes", "min_samples_leaf") else float(best[k])) for k in GRID}
json.dump(best_params, open("best_params.json", "w"), indent=1)
print("best:", best_params)
print(f"PR-AUC {best['PR-AUC']} (before tuning: 0.582)")

best: {'learning_rate': 0.05, 'max_leaf_nodes': 127, 'min_samples_leaf': 20, 'l2_regularization': 1.0, 'max_features': 0.5}
PR-AUC 0.597 (before tuning: 0.582)
